# 👕 AI Product Intelligence System

# Task 3: Reverse Product Search

### Submitted By

**Name:** Ede Dinesh Venkata Pavan Kumar

**College:** ACE Engineering College

**Branch:** Information Technology

**GenAI Bootcamp – Day 2**

# 📌 Problem Statement

Customers often know what they want but struggle to find the most relevant products using keyword-based search.

Traditional search engines rely on exact keyword matching and may miss semantically similar products.

The objective of this task is to build an AI-powered Reverse Product Search system that understands the user's query using semantic embeddings and retrieves the most relevant fashion products.

# 🎯 Objective

Develop an intelligent search system that:

- Accepts a natural language query.
- Converts the query into semantic embeddings.
- Compares it with product embeddings.
- Retrieves the Top-K most relevant fashion products.
- Displays matching product images and details.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*50)
print("DEVICE CONFIGURATION")
print("="*50)

print("Device :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

DEVICE CONFIGURATION
Device : cuda
GPU : Tesla T4


# 📂 Dataset

Dataset Used:

Fashion Product Images Dataset

Contents

- styles.csv
- images/

Total Products

44,000+

The dataset contains fashion product metadata and images used for semantic search.

In [3]:
dataset_path = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset"

styles_path = os.path.join(dataset_path, "styles.csv")

df = pd.read_csv(styles_path, on_bad_lines="skip")

print("Dataset Loaded Successfully")
print(df.shape)

df.head()

Dataset Loaded Successfully
(44424, 10)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [4]:
df = df[
    [
        "id",
        "productDisplayName",
        "masterCategory",
        "subCategory",
        "articleType",
        "baseColour",
        "gender",
        "season",
        "usage"
    ]
]

df = df.dropna()

print("Clean Dataset Shape :", df.shape)

Clean Dataset Shape : (44077, 9)


# 🤖 AI Model

## Sentence Transformer

The Reverse Product Search system uses the **SentenceTransformer (all-MiniLM-L6-v2)** model to convert product descriptions and user queries into semantic embeddings.

Instead of relying on exact keyword matching, semantic embeddings capture the meaning of the query, allowing the system to retrieve the most relevant fashion products.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("="*60)
print("AI MODEL LOADED SUCCESSFULLY")
print("="*60)
print("Model : all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AI MODEL LOADED SUCCESSFULLY
Model : all-MiniLM-L6-v2


# 📝 Product Representation

Each product is converted into a descriptive sentence by combining multiple attributes.

This richer representation improves semantic understanding during product search.

In [6]:
df["product_text"] = (
    df["productDisplayName"].fillna("") + " | " +
    df["masterCategory"].fillna("") + " | " +
    df["subCategory"].fillna("") + " | " +
    df["articleType"].fillna("") + " | " +
    df["baseColour"].fillna("") + " | " +
    df["gender"].fillna("") + " | " +
    df["usage"].fillna("")
)

df[["productDisplayName", "product_text"]].head()

,productDisplayName,product_text
0,Turtle Check Men Navy Blue Shirt,Turtle Check Men Navy Blue Shirt | Apparel | T...
1,Peter England Men Party Blue Jeans,Peter England Men Party Blue Jeans | Apparel |...
2,Titan Women Silver Watch,Titan Women Silver Watch | Accessories | Watch...
3,Manchester United Men Solid Black Track Pants,Manchester United Men Solid Black Track Pants ...
4,Puma Men Grey T-shirt,Puma Men Grey T-shirt | Apparel | Topwear | Ts...


# 🧠 Product Embeddings

The Sentence Transformer converts every product description into a numerical embedding vector.

These embeddings are stored and later compared with the user's query embedding.

In [7]:
product_embeddings = model.encode(
    df["product_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("="*60)
print("EMBEDDINGS GENERATED")
print("="*60)
print("Embedding Shape :", product_embeddings.shape)

Batches:   0%|          | 0/689 [00:00<?, ?it/s]

EMBEDDINGS GENERATED
Embedding Shape : (44077, 384)


# 🔍 Reverse Product Search

The user enters a natural language query.

The system converts the query into an embedding and retrieves the most semantically similar fashion products using cosine similarity.

In [8]:
def reverse_product_search(query, top_k=5):

    # Generate query embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Compute cosine similarity
    similarity_scores = cosine_similarity(
        query_embedding,
        product_embeddings
    )[0]

    # Get top results
    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices].copy()
    results["Similarity Score"] = similarity_scores[top_indices]

    return results

In [9]:
image_folder = os.path.join(dataset_path, "images")

def display_results(results):

    plt.figure(figsize=(15,10))

    for i, (_, row) in enumerate(results.iterrows()):

        plt.subplot(1, len(results), i+1)

        image_path = os.path.join(
            image_folder,
            str(row["id"]) + ".jpg"
        )

        image = Image.open(image_path)

        plt.imshow(image)
        plt.axis("off")

        plt.title(
            f"{row['articleType']}\nScore: {row['Similarity Score']:.2f}",
            fontsize=8
        )

    plt.tight_layout()
    plt.show()

    display(
        results[
            [
                "productDisplayName",
                "articleType",
                "baseColour",
                "gender",
                "Similarity Score"
            ]
        ]
    )

# 🔎 Search Interface

The user enters a natural language query.

Examples:

- blue casual shirt
- black handbag
- men's sports shoes
- women floral kurta
- red t-shirt

The system retrieves the most semantically similar products.

In [10]:
query = input("🔍 Enter a fashion product to search: ")

print("=" * 60)
print("USER QUERY")
print("=" * 60)

print(query)

StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:
results = reverse_product_search(query)

results

In [ ]:
display_results(results)

# 🧪 Multiple Search Examples

In [ ]:
queries = [

    "blue casual shirt",

    "black handbag",

    "red kurta",

    "sports shoes",

    "women jeans"

]

for query in queries:

    print("="*60)

    print("Query :", query)

    print("="*60)

    results = reverse_product_search(query)

    display(results.head(5))

# 📈 Search Performance

In [ ]:
import time

query = "blue casual shirt"

start = time.time()

results = reverse_product_search(query)

end = time.time()

print("="*50)

print("Search Time")

print("="*50)

print(f"{end-start:.4f} seconds")

# 📊 AI Workflow

```text
User Query

↓

Sentence Transformer

↓

Query Embedding

↓

Cosine Similarity

↓

Product Embeddings

↓

Top K Search Results

↓

Fashion Products
```

# 🏗 System Architecture

```text
                User Query
                     │
                     ▼
      Sentence Transformer Model
                     │
                     ▼
          Semantic Query Embedding
                     │
                     ▼
      Product Embedding Database
                     │
                     ▼
         Cosine Similarity Search
                     │
                     ▼
         Top Matching Products
```

# 💡 Technologies Used

- Python

- Pandas

- NumPy

- Sentence Transformers

- Scikit-Learn

- Matplotlib

- PIL

- Kaggle Notebook

- GPU (Tesla T4)

# ✅ Conclusion

An AI-powered Reverse Product Search system was successfully developed.

The system converts user queries into semantic embeddings using Sentence Transformers and retrieves the most relevant fashion products using cosine similarity.

Unlike traditional keyword search, this semantic search engine understands the meaning of the query and provides more relevant results.

This approach improves product discovery in e-commerce applications and demonstrates the practical use of Generative AI for intelligent product search.

# 🎯 Future Improvements

- Personalized product recommendations

- CLIP-based image and text search

- Voice search

- Multilingual search

- Real-time indexing using FAISS

- Hybrid image + text retrieval